In [64]:
"""
Training script for the sparse-view CT U-Net (image-domain, FBPConvNet-style).

Images are paired per patient by the NUMBER in the filename
(image_0 <-> slice_00, image_1 <-> slice_01, ...).

The split is done at PATIENT level (no slices of the same patient appear in
more than one split), which avoids data leakage.
"""

'\nTraining script for the sparse-view CT U-Net (image-domain, FBPConvNet-style).\n\nImages are paired per patient by the NUMBER in the filename\n(image_0 <-> slice_00, image_1 <-> slice_01, ...).\n\nThe split is done at PATIENT level (no slices of the same patient appear in\nmore than one split), which avoids data leakage.\n'

In [65]:
import random
import re
from pathlib import Path

In [66]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch.utils.data import DataLoader, Dataset

In [67]:
import os
p = "/content/drive/MyDrive/ct_project/IPPR_CT_Reconstruction"
print(os.path.exists(p))
print(os.listdir(p) if os.path.exists(p) else "nicht erreichbar")

True
['README.md', '.gitignore', 'requirements.txt', 'unet_ct.py', 'train_unet.py', 'unet_ct.ipynb', 'merge.py', 'kombiniert.ipynb', 'best_unet.pt', 'Log', 'Dataset_Evaluation', '.git', '__pycache__', 'MGST_5_IPPR_CT_Reconstruction', 'full_output', 'train_unet_npy.ipynb', 'train_unet.ipynb', 'data', 'full_reconstruction', 'sparse_reconstruction', 'unet_reconstructions', 'unzip_sinoct_to_drive.ipynb', 'train_unet_20train_10recon.ipynb', 'train_unet_collab_fixed.ipynb', 'train_unet_collab.ipynb']


In [68]:
from google.colab import drive
drive.mount("/content/drive")

!find /content/drive/MyDrive -maxdepth 5 -type d -name "full_reconstruction*"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ct_project/IPPR_CT_Reconstruction/full_reconstruction


In [69]:
FULL_DIR = Path(p) / "full_reconstruction"
SPARSE_DIR = Path(p) / "sparse_reconstruction"

In [70]:
IMG_SIZE = 256          # must be divisible by 16 (4 pooling stages in the U-Net)
BATCH_SIZE = 4
EPOCHS = 50
LR = 1e-4
SEED = 42

In [71]:
MAX_SERIES = 30           # total number of series used (sorted by series number)
N_TRAIN_PATIENTS = 20     # first 20 patients -> training
N_RECON_PATIENTS = 10     # last 10 patients -> reconstruction + saving only
N_VAL_PATIENTS = 0        # optional: number of the 20 training patients held out for validation
                          # (0 = all 20 used for training, the state after the last epoch is saved)

RECON_DIR = Path(p) / "unet_reconstructions"   # output folder for the reconstructed images

# Normalization: None = computed automatically in main() from the (full-view) training images.
# After the first run you can enter the printed values here (e.g. NORM_MIN = -0.02),
# so they are not recomputed and best_unet.pt can be reused later with the same values.
NORM_MIN = None
NORM_MAX = None

In [72]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [73]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [74]:
class DoubleConv(nn.Module):
    """(Conv -> BatchNorm -> ReLU) x 2"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """U-Net with 4 down- / 4 upsampling stages and residual learning:
    the network predicts the artifact pattern, which is subtracted from the input."""

    def __init__(self, in_channels=1, out_channels=1, base_channels=32):
        super().__init__()
        c = base_channels

        # Encoder
        self.enc1 = DoubleConv(in_channels, c)
        self.enc2 = DoubleConv(c, c * 2)
        self.enc3 = DoubleConv(c * 2, c * 4)
        self.enc4 = DoubleConv(c * 4, c * 8)
        self.pool = nn.MaxPool2d(kernel_size=2)

        # Bottleneck
        self.bottleneck = DoubleConv(c * 8, c * 16)

        # Decoder
        self.up4 = nn.ConvTranspose2d(c * 16, c * 8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(c * 16, c * 8)
        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(c * 8, c * 4)
        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(c * 4, c * 2)
        self.up1 = nn.ConvTranspose2d(c * 2, c, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(c * 2, c)

        self.out_conv = nn.Conv2d(c, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        residual = self.out_conv(d1)
        return x - residual

In [75]:
# Pairing images
def file_index(path):
    """Extract the (last) number in a filename: 'slice_07.png' -> 7."""
    numbers = re.findall(r"\d+", path.stem)
    return int(numbers[-1]) if numbers else None

In [76]:
def collect_pairs(series_name):
    """Return a list of (sparse_path, full_path) for one series
    (e.g. 'series_1009'), matched by the slice number in the filename.
    The number of slices differs between series, so only slices that exist
    in BOTH folders are used."""
    full_files, sparse_files = {}, {}
    for p in (FULL_DIR / series_name).glob("slice_*.png"):
        idx = file_index(p)
        if idx is not None:
            full_files[idx] = p
    for p in (SPARSE_DIR / series_name).glob("slice_*.png"):
        idx = file_index(p)
        if idx is not None:
            sparse_files[idx] = p

    common = sorted(set(full_files) & set(sparse_files))
    if len(common) != len(full_files) or len(common) != len(sparse_files):
        print(f"[warning] {series_name}: {len(full_files)} full, "
              f"{len(sparse_files)} sparse, {len(common)} matched")
    return [(sparse_files[i], full_files[i]) for i in common]

In [77]:
def load_npy(path):
    # Load the PNG image, convert to grayscale (L) and normalize to float32 [0.0, 255.0] or [0.0, 1.0]
    # Typically, PNG files are 8-bit [0, 255]. We convert them to float32 arrays.
    with Image.open(path) as img:
        arr = np.array(img.convert('L')).astype(np.float32)
    arr = (arr - NORM_MIN) / (NORM_MAX - NORM_MIN)   # SAME constants for sparse AND full
    t = torch.from_numpy(arr)[None, None]            # (1, 1, H, W)
    t = F.interpolate(t, size=(IMG_SIZE, IMG_SIZE), mode="bilinear",
                      antialias=True, align_corners=False)
    return t[0]                                      # (1, H, W)


def compute_norm_constants(patient_names):
    """Min/max over all FULL-view training images (only training patients,
    so no information from the other patients flows into the normalization)."""
    lo, hi = np.inf, -np.inf
    for name in patient_names:
        for _, full_path in collect_pairs(name):
            with Image.open(full_path) as img:
                arr = np.array(img.convert('L')).astype(np.float32)
            lo = min(lo, float(arr.min()))
            hi = max(hi, float(arr.max()))
    return lo, hi

In [78]:
class CTPairDataset(Dataset):
    """Holds all (sparse, original) pairs of a list of patients in memory
    (small dataset, so this is fine and makes training fast)."""

    def __init__(self, patient_names, augment=False):
        self.augment = augment
        self.sparse, self.original = [], []
        for name in patient_names:
            for sparse_path, orig_path in collect_pairs(name):
                self.sparse.append(load_npy(sparse_path))
                self.original.append(load_npy(orig_path))

    def __len__(self):
        return len(self.sparse)

    def __getitem__(self, idx):
        x, y = self.sparse[idx], self.original[idx]
        if self.augment:
            # The SAME flip must be applied to input and target
            if random.random() < 0.5:
                x, y = torch.flip(x, dims=[2]), torch.flip(y, dims=[2])
            if random.random() < 0.5:
                x, y = torch.flip(x, dims=[1]), torch.flip(y, dims=[1])
        return x, y

In [79]:
# Patient-level split
def split_patients():
    # Only use series folders that exist in BOTH the full and the sparse directory
    full_series = {q.name for q in FULL_DIR.iterdir() if q.is_dir() and q.name.startswith("series_")}
    sparse_series = {q.name for q in SPARSE_DIR.iterdir() if q.is_dir() and q.name.startswith("series_")}
    missing = full_series ^ sparse_series
    if missing:
        print(f"[warning] series only present in one directory (ignored): {sorted(missing)}")

    # Fixed order by series number (no shuffle): first = training, last = reconstruction
    patients = sorted(full_series & sparse_series, key=lambda s: int(s.split("_")[1]))
    if MAX_SERIES is not None:
        patients = patients[:MAX_SERIES]
    print(f"Found {len(patients)} series")

    train_all = patients[:N_TRAIN_PATIENTS]
    recon = patients[N_TRAIN_PATIENTS:N_TRAIN_PATIENTS + N_RECON_PATIENTS]

    if N_VAL_PATIENTS > 0:
        val = train_all[-N_VAL_PATIENTS:]
        train = train_all[:-N_VAL_PATIENTS]
    else:
        val, train = [], train_all

    print(f"Train patients ({len(train)}): {train}")
    print(f"Val patients   ({len(val)}): {val}")
    print(f"Recon patients ({len(recon)}): {recon}")
    return train, val, recon

In [80]:
# Training / evaluation
def run_epoch(model, loader, loss_fn, optimizer=None):
    """One pass over the loader. Trains if an optimizer is given, else evaluates."""
    training = optimizer is not None
    model.train(training)
    total = 0.0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)
            loss = loss_fn(pred, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item() * x.size(0)
    return total / len(loader.dataset)

In [81]:
@torch.no_grad()
def evaluate_metrics(model, dataset):
    """PSNR / SSIM of (a) the raw sparse FBP input and (b) the U-Net output,
    both compared to the ground truth. (a) is your FBP baseline."""
    model.eval()
    rows = {"input": {"psnr": [], "ssim": []}, "unet": {"psnr": [], "ssim": []}}
    for x, y in dataset:
        pred = model(x.unsqueeze(0).to(DEVICE)).squeeze().clamp(0, 1).cpu().numpy()
        x_np, y_np = x.squeeze().numpy(), y.squeeze().numpy()
        for name, img in (("input", x_np), ("unet", pred)):
            rows[name]["psnr"].append(peak_signal_noise_ratio(y_np, img, data_range=1.0))
            rows[name]["ssim"].append(structural_similarity(y_np, img, data_range=1.0))
    for name, r in rows.items():
        print(f"{name:>6}:  PSNR {np.mean(r['psnr']):.2f} dB   SSIM {np.mean(r['ssim']):.4f}")

In [82]:
@torch.no_grad()
def reconstruct_and_save(model, patient_names, out_dir):
    """Reconstruct all slices of the given patients with the trained U-Net
    and save them (no training, no metrics).

    Two files are written per slice:
      <out_dir>/<series>/slice_XX.npy  -> float32, back in the original value range (NORM_MIN..NORM_MAX)
      <out_dir>/<series>/slice_XX.png  -> 8-bit preview (value range 0..1 mapped to 0..255)
    The images have size IMG_SIZE x IMG_SIZE (same as during training).
    """
    model.eval()
    out_dir = Path(out_dir)
    n_saved = 0
    for name in patient_names:
        series_dir = out_dir / name
        series_dir.mkdir(parents=True, exist_ok=True)
        pairs = collect_pairs(name)
        for start in range(0, len(pairs), BATCH_SIZE):
            batch_pairs = pairs[start:start + BATCH_SIZE]
            x = torch.stack([load_npy(sp) for sp, _ in batch_pairs]).to(DEVICE)
            pred = model(x).clamp(0, 1).cpu().numpy()[:, 0]          # (B, H, W), in [0, 1]
            for (sparse_path, _), img in zip(batch_pairs, pred):
                stem = sparse_path.stem                              # e.g. 'slice_07'
                np.save(series_dir / f"{stem}.npy",
                        (img * (NORM_MAX - NORM_MIN) + NORM_MIN).astype(np.float32))
                Image.fromarray((img * 255).round().astype(np.uint8)).save(series_dir / f"{stem}.png")
                n_saved += 1
        print(f"{name}: {len(pairs)} Slices gespeichert")
    print(f"\nFertig: {n_saved} rekonstruierte Bilder in {out_dir}")


In [83]:
def main():
    global NORM_MIN, NORM_MAX
    set_seed(SEED)
    train_p, val_p, recon_p = split_patients()

    # ---------- 0) Normalization constants (computed from the training images only) ----------
    if NORM_MIN is None or NORM_MAX is None:
        NORM_MIN, NORM_MAX = compute_norm_constants(train_p)
    print(f"NORM_MIN = {NORM_MIN:.6f}, NORM_MAX = {NORM_MAX:.6f}")

    # ---------- 1) Training on the first 20 patients ----------
    train_ds = CTPairDataset(train_p, augment=True)
    val_ds = CTPairDataset(val_p) if val_p else None
    print(f"Images -> train: {len(train_ds)}, val: {len(val_ds) if val_ds else 0}")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE) if val_ds else None

    model = UNet(in_channels=1, out_channels=1, base_channels=32).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.L1Loss()

    best_val = float("inf")
    for epoch in range(1, EPOCHS + 1):
        tr = run_epoch(model, train_loader, loss_fn, optimizer)
        if val_loader is not None:
            va = run_epoch(model, val_loader, loss_fn)
            flag = ""
            if va < best_val:      # keep the model that does best on validation
                best_val = va
                torch.save(model.state_dict(), "best_unet.pt")
                flag = "  <- saved"
            print(f"Epoch {epoch:03d} | train L1 {tr:.5f} | val L1 {va:.5f}{flag}")
        else:
            torch.save(model.state_dict(), "best_unet.pt")   # without validation: always keep the latest state
            print(f"Epoch {epoch:03d} | train L1 {tr:.5f}")

    model.load_state_dict(torch.load("best_unet.pt", map_location=DEVICE))

    # ---------- 2) Last 10 patients: reconstruct + save only ----------
    print("\n=== Rekonstruktion der letzten Patienten ===")
    reconstruct_and_save(model, recon_p, RECON_DIR)

In [84]:
if __name__ == "__main__":
    main()

Found 30 series
Train patients (20): ['series_1009', 'series_1010', 'series_1011', 'series_1012', 'series_1013', 'series_1014', 'series_1015', 'series_1016', 'series_1017', 'series_1018', 'series_1019', 'series_1020', 'series_1021', 'series_1022', 'series_1023', 'series_1024', 'series_1025', 'series_1026', 'series_1027', 'series_1028']
Val patients   (0): []
Recon patients (10): ['series_1029', 'series_1030', 'series_1031', 'series_1032', 'series_1033', 'series_1034', 'series_1035', 'series_1036', 'series_1037', 'series_1038']
NORM_MIN = 0.000000, NORM_MAX = 255.000000
Images -> train: 780, val: 0
Epoch 001 | train L1 0.03601


KeyboardInterrupt: 